In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

SAMPLING_SIZE = 128

In [ ]:
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import dijkstra
from scipy.spatial.transform import Rotation

#from isomap_master.isomap import make_adjacency, isomap, plot_graph
from sklearn.decomposition import PCA

from Utility import *

import time
import os

#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
DPath = "/home/pgh/2026CapstoneDesign_08_01/Content/03Python/AppearanceManifold/Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_BaseColor.PNG"
SPath = "/home/pgh/2026CapstoneDesign_08_01/Content/03Python/AppearanceManifold/Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_Specular.PNG"
RPath = "/home/pgh/2026CapstoneDesign_08_01/Content/03Python/AppearanceManifold/Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_Roughness.PNG"
D,S,R = load_and_downsample(DPath, SPath, RPath, SAMPLING_SIZE)

X = combine_to_7d_tensor(D,S,R) # 2차원 이미지 형태
FX = combine_to_7d(D,S,R)       # 1차원 배열

#visualize_sample(X)

In [ ]:
from KNN import Get_KNN_graph, Get_KNN_Graph_Adaptive, Get_KNN_Graph_Adaptive_Streaming
from Geodesic import Get_Geodesic_Distance_Matrix, Get_Geodesic_Distance_Matrix_GPU
from MDS import Get_MDS_graph, visualize_MDS_Graph, visualize_MDS_Graph_Edges, visualize_MDS_Graph_Edges_2D

In [ ]:
Z = normalize_features(FX)
print(Z.shape[0])
print(Z)

In [ ]:
src, dst, wei, wea = Get_KNN_Graph_Adaptive(Z, K=8)
#src, dst, wei, wea = Get_KNN_Graph_Adaptive_Streaming(Z, K=4)
print(src, src.shape)
print(dst, dst.shape)
print(wei, wei.shape)
print(wea)

In [ ]:
N = Z.shape[0]
#N = wei.shape[0]
D = Get_Geodesic_Distance_Matrix(src, dst, wei, N)

print(D.shape)
print(D)



In [ ]:
#D2 = Get_Geodesic_Distance_Matrix_GPU(src, dst, wei, wea.shape[0])
#print(D2.shape)
#print(D2)

In [ ]:
M = Get_MDS_graph(D, 3)
print(M.shape)
print(M)

In [ ]:
visualize_MDS_Graph(M,Z)

In [ ]:
visualize_MDS_Graph_Edges(M, src, dst, Z)

In [ ]:
visualize_MDS_Graph_Edges_2D(M, src, dst, Z)

In [ ]:
from Trajectory import * 

In [ ]:
start_idx = np.argmin(wea)
end_idx = np.argmax(wea)

trajectory_points, trajectory_nodes = (
    build_weathering_trajectory(
        embedded=M,
        edge_src=src,
        edge_dst=dst,
        edge_weight=wei,
        start_idx=start_idx,
        end_idx=end_idx
    )
)

trajectory = resample_trajectory(
    trajectory_points,
    num_points=SAMPLING_SIZE * 2
)

visualize_trajectory(M, trajectory)



In [ ]:
from Utility import visualize_MDS_Graph_With_Trajectory

visualize_MDS_Graph_With_Trajectory(
    Z=M,
    edge_src=src,
    edge_dst=dst,
    trajectory=trajectory
)

In [ ]:
from Reconstructor import *

In [ ]:
steps = build_weathering_sequence_semantic(FX, M, trajectory, W=SAMPLING_SIZE, H=SAMPLING_SIZE, max_step=30)
visualize_weathering_sequence(steps)

In [ ]:
from LoggerClass import evaluate_all_sequence
evaluate_all_sequence(src, dst, wei, Z, D, M, trajectory, "Texture_Eval")